# EnergyGuard AI — Reproducible Workflow

This notebook follows the project PRD: audit the two datasets independently, preprocess them, train the four model instances, and inspect held-out evaluation. Dataset files are not distributed with this repository; place them in `data/raw/` first.

**Important:** theft-risk outputs are screening indicators only, not proof of theft.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from src.config import RAW_DIR, PROCESSED_DIR, REPORT_DIR, MODEL_DIR
from src.data_pipeline import run_audit, prepare_uci, prepare_sgcc

print('Raw data directory:', RAW_DIR.resolve())
print('Files:', [p.name for p in RAW_DIR.iterdir()])

## 1. Dataset audit

The UCI dataset is used for forecasting and unsupervised anomaly detection. SGCC is used separately for consumer-level binary classification. The tables are never merged.

In [ ]:
audit = run_audit()
print(json.dumps(audit, indent=2, default=str))

## 2. UCI preprocessing and feature engineering

The pipeline converts minute-average active power (kW) into hourly energy (kWh), retains hours with at least 54 valid minute readings, creates lag/rolling/calendar features, and sets the next-hour target. Splitting is chronological inside the training module.

In [ ]:
uci = prepare_uci()
uci.head()

In [ ]:
uci.describe(include='all')

## 3. SGCC consumer feature engineering

Daily readings are ordered chronologically where date-formatted column names are available. The pipeline creates consumer-level statistics and leaves the target FLAG unchanged. Missing-value imputation is fitted inside the training pipeline only.

In [ ]:
sgcc, sgcc_features = prepare_sgcc()
print('Feature columns:', sgcc_features)
print('Class counts:')
display(sgcc['FLAG'].value_counts().sort_index())
sgcc.head()

## 4. Train and evaluate models

Run the training functions after confirming that both datasets are available. The training module performs chronological UCI splits, consumer-level stratified SGCC splits, training-only imputation, validation-based model/threshold selection, and final held-out test evaluation.

In [ ]:
from src.train import train_forecasting, train_anomaly, train_theft

forecast_results = train_forecasting()
anomaly_results = train_anomaly()
theft_results = train_theft()
print('Forecast:', json.dumps(forecast_results, indent=2, default=float))
print('Anomaly:', json.dumps(anomaly_results, indent=2, default=float))
print('Theft:', json.dumps(theft_results, indent=2, default=float))

## 5. Inspect generated artifacts

Metrics and plots are saved under `reports/`; fitted model pipelines are saved under `models/`. No performance values are pre-filled in this notebook: results are produced only after training on the actual datasets.

In [ ]:
print('Reports:', sorted(p.name for p in REPORT_DIR.glob('*')))
print('Models:', sorted(p.name for p in MODEL_DIR.glob('*')))
for metric_file in REPORT_DIR.glob('*_metrics.json'):
    print('\n', metric_file.name)
    print(metric_file.read_text())